1) Run cells 1→2→3→4→5
2) Run cell 6 to keep streaming continuously
3) Press ■ to stop

## 1 — Health Checks

In [1]:
import socket
import time


def wait_for_tcp(host, port, label, timeout=120):
    print(f"Waiting for {label} ({host}:{port})...", end="", flush=True)
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            socket.create_connection((host, port), timeout=2).close()
            print(" OK")
            return
        except OSError:
            print(".", end="", flush=True)
            time.sleep(3)
    raise TimeoutError(f"{label} not ready after {timeout}s")


wait_for_tcp("postgres", 5432, "PostgreSQL")
wait_for_tcp("minio", 9000, "MinIO")
wait_for_tcp("hive-metastore", 9083, "Hive Metastore")
wait_for_tcp("spark-master", 7077, "Spark Master")
wait_for_tcp("kafka", 9092, "Kafka Broker")

print("\nAll required services ready.")

Waiting for PostgreSQL (postgres:5432)... OK
Waiting for MinIO (minio:9000)... OK
Waiting for Hive Metastore (hive-metastore:9083)... OK
Waiting for Spark Master (spark-master:7077)... OK
Waiting for Kafka Broker (kafka:9092)... OK

All required services ready.


## 2 — Spark Session

In [2]:
import os
import time
from pyspark.sql import SparkSession

KAFKA_BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:9092")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_KEY = os.getenv("MINIO_ROOT_USER", "minio")
MINIO_SECRET = os.getenv("MINIO_ROOT_PASSWORD", "minio123")

DELTA_BASE = "s3a://lakehouse/staging"
CHECKPOINT_BASE = "s3a://lakehouse/checkpoints"
SCHEMA_DIR = os.getenv("SCHEMA_DIR", "/schemas")

DB_HOST = os.getenv("POSTGRES_HOST", "postgres")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB", "thelook")
DB_USER = os.getenv("POSTGRES_USER", "admin")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD", "admin123")

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()
    time.sleep(2)

spark = (
    SparkSession.builder
    .appName("TheLookStreaming")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark session ready.")

Spark session ready.


## 3 — Schemas + Decode Helpers

In [3]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

# ── CDC Envelope schema (Debezium SMT unwraps to this) ──────────────────────
CDC_ENVELOPE_SCHEMA = T.StructType([
    T.StructField("before", T.StringType(), True),
    T.StructField("after",  T.StringType(), True),
    T.StructField("op",     T.StringType(), True),
    T.StructField("ts_ms",  T.StringType(), True),
])

# ── Spark StructTypes — must match exact Kafka wire types from Debezium ──────
# PostgreSQL UUID → JSON string, BIGINT → JSON number
USERS_SCHEMA = T.StructType([
    T.StructField("id",             T.StringType(), True),
    T.StructField("first_name",     T.StringType(), True),
    T.StructField("last_name",      T.StringType(), True),
    T.StructField("email",          T.StringType(), True),
    T.StructField("age",            T.IntegerType(), True),
    T.StructField("gender",        T.StringType(), True),
    T.StructField("street_address",T.StringType(), True),
    T.StructField("postal_code",    T.StringType(), True),
    T.StructField("city",           T.StringType(), True),
    T.StructField("state",          T.StringType(), True),
    T.StructField("country",        T.StringType(), True),
    T.StructField("latitude",       T.DoubleType(), True),
    T.StructField("longitude",      T.DoubleType(), True),
    T.StructField("traffic_source", T.StringType(), True),
    T.StructField("created_at",     T.StringType(), True),
    T.StructField("updated_at",     T.StringType(), True),
])

ORDERS_SCHEMA = T.StructType([
    T.StructField("id",           T.StringType(), True),   # UUID
    T.StructField("user_id",      T.StringType(), True),   # UUID
    T.StructField("status",       T.StringType(), True),
    T.StructField("num_of_items", T.IntegerType(), True),
    T.StructField("created_at",   T.StringType(), True),
    T.StructField("updated_at",   T.StringType(), True),
    T.StructField("returned_at",  T.StringType(), True),
    T.StructField("shipped_at",   T.StringType(), True),
    T.StructField("delivered_at", T.StringType(), True),
    T.StructField("cancelled_at", T.StringType(), True),
])

ORDER_ITEMS_SCHEMA = T.StructType([
    T.StructField("id",           T.StringType(), True),   # UUID
    T.StructField("order_id",     T.StringType(), True),   # UUID
    T.StructField("product_id",   T.LongType(), True),      # BIGINT
    T.StructField("status",       T.StringType(), True),
    T.StructField("quantity",     T.IntegerType(), True),
    T.StructField("sale_price",   T.DoubleType(), True),
    T.StructField("created_at",   T.StringType(), True),
    T.StructField("updated_at",   T.StringType(), True),
    T.StructField("shipped_at",   T.StringType(), True),
    T.StructField("delivered_at", T.StringType(), True),
    T.StructField("returned_at",  T.StringType(), True),
    T.StructField("cancelled_at", T.StringType(), True),
])

EVENTS_SCHEMA = T.StructType([
    T.StructField("id",              T.StringType(), True),   # UUID
    T.StructField("user_id",         T.StringType(), True),   # UUID
    T.StructField("sequence_number", T.IntegerType(), True),
    T.StructField("session_id",      T.StringType(), True),
    T.StructField("ip_address",      T.StringType(), True),
    T.StructField("city",            T.StringType(), True),
    T.StructField("state",           T.StringType(), True),
    T.StructField("postal_code",     T.StringType(), True),
    T.StructField("browser",         T.StringType(), True),
    T.StructField("traffic_source",  T.StringType(), True),
    T.StructField("uri",             T.StringType(), True),
    T.StructField("event_type",     T.StringType(), True),
    T.StructField("created_at",     T.StringType(), True),
])

# Static tables are bootstrapped via JDBC — types come from PostgreSQL (no CDC involved)
REF_PRODUCTS_SCHEMA = T.StructType([
    T.StructField("id",                     T.LongType(), True),      # BIGINT
    T.StructField("cost",                  T.DoubleType(), True),
    T.StructField("category",              T.StringType(), True),
    T.StructField("name",                   T.StringType(), True),
    T.StructField("brand",                  T.StringType(), True),
    T.StructField("retail_price",          T.DoubleType(), True),
    T.StructField("department",            T.StringType(), True),
    T.StructField("sku",                   T.StringType(), True),
    T.StructField("distribution_center_id", T.LongType(), True),   # BIGINT
])

REF_DIST_CENTERS_SCHEMA = T.StructType([
    T.StructField("id",        T.LongType(), True),      # BIGINT
    T.StructField("name",      T.StringType(), True),
    T.StructField("latitude",   T.DoubleType(), True),
    T.StructField("longitude", T.DoubleType(), True),
])

print("StructType schemas ready.")

StructType schemas ready.


## 4 — Schemas

In [ ]:
import subprocess

TRINO = os.getenv("TRINO_URL", "http://trino:8080")

def trino(sql):
    """Run Trino SQL via CLI."""
    cmd = ["trino", "--server", TRINO, "--execute", sql, "--progress"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  WARN: {result.stderr.strip()}")
    return result.returncode == 0

def bootstrap(src_table, tgt_path, table_name):
    """Bootstrap static reference table from PostgreSQL to Delta Lake."""
    df = (
        spark.read.format("jdbc")
        .option("url", f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}")
        .option("dbtable", f"public.{src_table}")
        .option("user", DB_USER)
        .option("password", DB_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .load()
    )
    if "operation" not in df.columns:
        df = df.withColumn("operation", F.lit("r"))
    if "event_ts_ms" not in df.columns:
        df = df.withColumn("event_ts_ms", (F.unix_timestamp(F.current_timestamp()) * 1000).cast("long"))

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(tgt_path)
    print(f"Bootstrapped: {table_name} -> {tgt_path} ({df.count()} rows)")


# Create staging schema + register static tables via Trino
trino("CREATE SCHEMA IF NOT EXISTS delta.staging")
print("Schema staging created.")

bootstrap("products", f"{DELTA_BASE}/ref_products", "ref_products")
trino(f"CALL delta.register_table('staging', 'ref_products', '{DELTA_BASE}/ref_products')")
print("Registered: staging.ref_products")

bootstrap("dist_centers", f"{DELTA_BASE}/ref_dist_centers", "ref_dist_centers")
trino(f"CALL delta.register_table('staging', 'ref_dist_centers', '{DELTA_BASE}/ref_dist_centers')")
print("Registered: staging.ref_dist_centers")

print("Static ref tables ready.")


## 5 — Bootstrap + Register Static Tables

In [ ]:
# Cell 5 intentionally left empty — static tables are registered by cell 4.
# CDC tables are registered by cell 6 after the streaming query starts.
print("HMS registration done in cell 4 (static) and cell 6 (CDC).")

## 6 — Start CDC Streams (continuous)

In [ ]:
# Tune streaming cadence
spark.conf.set("spark.sql.shuffle.partitions", "8")

# Map: Kafka topic -> (table_name, table_location, StructType)
CDC_TOPICS = {
    "thelook.public.users":       ("users",       f"{DELTA_BASE}/users",       USERS_SCHEMA),
    "thelook.public.orders":       ("orders",       f"{DELTA_BASE}/orders",       ORDERS_SCHEMA),
    "thelook.public.order_items":  ("order_items",  f"{DELTA_BASE}/order_items",  ORDER_ITEMS_SCHEMA),
    "thelook.public.events":       ("events",       f"{DELTA_BASE}/events",       EVENTS_SCHEMA),
}

# ── Start streaming queries ───────────────────────────────────────────────────
queries = []
for topic, (table, table_loc, table_schema) in CDC_TOPICS.items():
    # ── 1. Read raw Kafka messages ─────────────────────────────────────────────
    raw = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
        .option("subscribe", topic)
        .option("kafka.group.id", "spark-streaming")
        .option("startingOffsets", "earliest")
        .option("failOnDataLoss", "false")
        .option("maxOffsetsPerTrigger", "10000")
        .load()
    )

    # ── 2. Sakila-Lakehouse pattern: two-layer from_json ───────────────────────
    # Layer 1: parse CDC envelope JSON
    envelope_df = raw.selectExpr(
        "CAST(value AS STRING) AS _raw",
        "CAST(key AS STRING) AS kafka_key",
        "CAST(timestamp AS TIMESTAMP) AS kafka_ts",
        "partition", "offset"
    ).select(
        F.from_json(F.col("_raw"), CDC_ENVELOPE_SCHEMA).alias("envelope"),
        F.col("kafka_key"),
        F.col("kafka_ts"),
        F.col("partition"),
        F.col("offset"),
    )

    # Layer 2: parse 'after' JSON → typed columns (schema-driven, no manual cast)
    enriched = envelope_df.select(
        F.from_json(F.col("envelope.after"), table_schema).alias("data"),
        F.col("envelope.op").alias("op"),
        F.col("envelope.ts_ms").alias("ts_ms"),
        F.col("kafka_key"),
        F.col("kafka_ts"),
        F.col("partition"),
        F.col("offset"),
    )

    # ── 3. Filter null rows (from old Avro corrupt messages before JSON switch) ─
    filtered = enriched.filter(F.col("data.id").isNotNull())

    # ── 4. Project final columns ────────────────────────────────────────────────
    data_cols = [F.col(f"data.{f.name}").alias(f.name) for f in table_schema.fields]
    final = filtered.select(*data_cols, "kafka_ts", "partition", "offset", "op", "ts_ms")

    # ── 5. Write to Delta with checkpoint ───────────────────────────────────────
    q = (
        final.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/{table}")
        .option("mergeSchema", "true")
        .trigger(processingTime="30 seconds")
        .start(f"{DELTA_BASE}/{table}")
    )
    queries.append(q)
    print(f"Stream started: {topic} -> {DELTA_BASE}/{table}")

# ── Register CDC tables in HMS via Trino CLI ───────────────────────────────────
print("\nRegistering CDC tables via Trino...")
for table, table_loc, _ in CDC_TOPICS.values():
    ok = trino(f"CALL delta.register_table('staging', '{table}', '{table_loc}')")
    status = "OK" if ok else "WARN"
    print(f"  [{status}] staging.{table}")

print("\nHMS registration done.")
print(f"\nAll {len(queries)} streams running (micro-batch every 30s). Press ■ to stop.")
spark.streams.awaitAnyTermination()
